In [0]:
%pip install great-expectations==0.17.23
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 32.7 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Import Libraries
import great_expectations as ge
from pyspark.sql.functions import col, count as sql_count


In [0]:
customers_bronze_path = "s3://travel-analytics-bronze/delta/bronze/customers/"
df_customers = spark.read.format("delta").load(customers_bronze_path)

print("=" * 80)
print("CUSTOMERS VALIDATION WITH GREAT EXPECTATIONS")
print("=" * 80)
print(f"Total records: {df_customers.count()}")
print("\n--- Schema ---")
df_customers.printSchema()

CUSTOMERS VALIDATION WITH GREAT EXPECTATIONS
Total records: 1000

--- Schema ---
root
 |-- _airbyte_ab_id: string (nullable = true)
 |-- _airbyte_emitted_at: timestamp (nullable = true)
 |-- _airbyte_additional_properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- _ab_cdc_lsn: double (nullable = true)
 |-- _ab_cdc_deleted_at: string (nullable = true)
 |-- _ab_cdc_updated_at: string (nullable = true)
 |-- id: long (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- birth_date: struct (nullable = true)
 |    |-- member0: date (nullable = true)
 |    |-- member1: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- family_name: string (nullable = true)
 |-- phone_number: string (nullable = true)



In [0]:
ge_df = ge.from_pandas(df_customers.toPandas())

print("\nRunning Great Expectations validation...")


Running Great Expectations validation...


In [0]:
# Define and Run Expectations

# Expectation 1: id NOT NULL
result1 = ge_df.expect_column_values_to_not_be_null(column="id")
print(f"✓ id NOT NULL: {result1['success']}")

# Expectation 2: id UNIQUE
result2 = ge_df.expect_column_values_to_be_unique(column="id")
print(f"✓ id UNIQUE: {result2['success']}")

# Expectation 3: first_name NOT NULL
result3 = ge_df.expect_column_values_to_not_be_null(column="first_name")
print(f"✓ first_name NOT NULL: {result3['success']}")

# Expectation 4: family_name NOT NULL
result4 = ge_df.expect_column_values_to_not_be_null(column="family_name")
print(f"✓ family_name NOT NULL: {result4['success']}")

# Expectation 5: phone_number NOT NULL
result5 = ge_df.expect_column_values_to_not_be_null(column="phone_number")
print(f"✓ phone_number NOT NULL: {result5['success']}")

✓ id NOT NULL: True
✓ id UNIQUE: True
✓ first_name NOT NULL: True
✓ family_name NOT NULL: True
✓ phone_number NOT NULL: True


In [0]:
print("\n--- Applying Validations ---")

# Start with all data
df_valid = df_customers
df_invalid_list = []

# Rule 1: id NOT NULL
df_invalid_null_id = df_valid.filter(col("id").isNull())
if df_invalid_null_id.count() > 0:
    df_invalid_list.append(df_invalid_null_id)
    print(f"  → Found {df_invalid_null_id.count()} rows with id = NULL")
df_valid = df_valid.filter(col("id").isNotNull())

# Rule 2: id UNIQUE
duplicates = df_valid.groupBy("id").agg(sql_count("*").alias("cnt")).filter(col("cnt") > 1)
duplicate_ids = [row.id for row in duplicates.collect()]
if duplicate_ids:
    df_invalid_dup = df_valid.filter(col("id").isin(duplicate_ids))
    df_invalid_list.append(df_invalid_dup)
    print(f"  → Found {df_invalid_dup.count()} rows with duplicate id")
    df_valid = df_valid.filter(~col("id").isin(duplicate_ids))

# Rule 3: first_name NOT NULL
df_invalid_first_name = df_valid.filter(col("first_name").isNull())
if df_invalid_first_name.count() > 0:
    df_invalid_list.append(df_invalid_first_name)
    print(f"  → Found {df_invalid_first_name.count()} rows with first_name = NULL")
df_valid = df_valid.filter(col("first_name").isNotNull())

# Rule 4: family_name NOT NULL
df_invalid_family_name = df_valid.filter(col("family_name").isNull())
if df_invalid_family_name.count() > 0:
    df_invalid_list.append(df_invalid_family_name)
    print(f"  → Found {df_invalid_family_name.count()} rows with family_name = NULL")
df_valid = df_valid.filter(col("family_name").isNotNull())

# Rule 5: phone_number NOT NULL
df_invalid_phone = df_valid.filter(col("phone_number").isNull())
if df_invalid_phone.count() > 0:
    df_invalid_list.append(df_invalid_phone)
    print(f"  → Found {df_invalid_phone.count()} rows with phone_number = NULL")
df_valid = df_valid.filter(col("phone_number").isNotNull())


--- Applying Validations ---


In [0]:
# STEP 8: Combine Invalid Records

if df_invalid_list:
    df_invalid = df_invalid_list[0]
    for df_temp in df_invalid_list[1:]:
        df_invalid = df_invalid.union(df_temp)
    df_invalid = df_invalid.distinct()
else:
    df_invalid = spark.createDataFrame([], df_customers.schema)

valid_count = df_valid.count()
invalid_count = df_invalid.count()
total_count = df_customers.count()

print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)
print(f"✅ Valid records:   {valid_count} ({valid_count/total_count*100:.2f}%)")
print(f"❌ Invalid records: {invalid_count} ({invalid_count/total_count*100:.2f}%)")



VALIDATION RESULTS
✅ Valid records:   1000 (100.00%)
❌ Invalid records: 0 (0.00%)


In [0]:
# STEP 9: Write Invalid Records to Quarantine

if invalid_count > 0:
    quarantine_path = "s3://travel-analytics-bronze/Quarantine/Customers"
    
    df_invalid.write \
        .format("parquet") \
        .mode("append") \
        .save(quarantine_path)
    
    print(f"\n❌ Invalid records sent to Quarantine: {quarantine_path}")
    print("\n--- Sample Invalid Records ---")
    df_invalid.show(10, truncate=False)
else:
    print(f"\n✅ All {valid_count} records passed validation!")

print("\n" + "=" * 80)
print("✅ VALIDATION COMPLETED!")
print("=" * 80)
print(f"Valid records remain in Bronze: {customers_bronze_path}")
if invalid_count > 0:
    print(f"Invalid records in Quarantine: s3://travel-analytics-bronze/Quarantine/Customers")


✅ All 1000 records passed validation!

✅ VALIDATION COMPLETED!
Valid records remain in Bronze: s3://travel-analytics-bronze/delta/bronze/customers/
